# Экспериментально: проверка STT на основе VOSK

Проверка работы преобразования текста в речь на основе Vosk

### Импорты, конфигурация

In [14]:
import os
import time
import wave
import json
import tempfile
import re
import tkinter as tk
from tkinter import filedialog
from typing import Final, Optional, Any
from pydub import AudioSegment
from vosk import Model, KaldiRecognizer

In [15]:
# --- Configuration ---
MODEL_PATH: Final[str] = "data/models/vosk-model-ru-0.42"

USE_VOCABULARY: bool = False
CUSTOM_VOCABULARY = [
    "проверка",
    "сложение",
    "вычитание",
    "конкатенация",
    "конъюнкция",
    "дизъюнкция",
    "импликация",
    "Лукасевич",
    "Лукашевич",
    "Клини",
    "Клин",
    "Клина",
    "эквиваленция",
    "исключающее",
    "или",
    "таблица",
    "для",
    "истинность",
    "истинности",
    "пожалуйста",
    "компьютер",
    "вывести",
    "[unk]"
]

In [16]:
def timer(func):
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter() # Record time before function execution
        result = func(*args, **kwargs)   # Call the original function
        end_time = time.perf_counter()   # Record time after function execution
        run_time = end_time - start_time
        print(f"'{func.__name__}' ran in {run_time:.4f} seconds")
        return result
    return wrapper

In [17]:
@timer
def load_model():
    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(f"Model directory '{MODEL_PATH}' not found. Please download it first.")
    return Model(MODEL_PATH)

model = load_model()

'load_model' ran in 116.4705 seconds


### Выбор файла

In [18]:
def select_file() -> Optional[str]:
    """Opens a file dialog to select a WAV file if no path is provided."""
    root = tk.Tk()
    root.withdraw()  # Hide the main tkinter window
    root.attributes("-topmost", True)  # Bring dialog to front
    file_path = filedialog.askopenfilename(
        title="Select Audio File",
        filetypes=[("Audio Files", "*.wav *.mp3 *.ogg *.flac"), ("All Files", "*.*")]
    )
    root.destroy()
    return file_path if file_path else None

# --- 1. Selection & Parameter Check ---
# Initialize AUDIO_FILE_PATH
AUDIO_FILE_PATH: Optional[str] = None

if not AUDIO_FILE_PATH:
    AUDIO_FILE_PATH = select_file()

if not AUDIO_FILE_PATH:
    raise FileNotFoundError("No file selected. Execution halted.")

AUDIO_FILE_PATH

'C:/Users/vadim/Music/audio_2026-04-01_17-25-19.wav'

### Загрузка аудиофайла, просмотр данных

In [19]:
@timer
def display_params(path: str) -> None:
    if not os.path.exists(path):
        print(f"Error: File {path} not found")
        return

    with wave.open(path, "rb") as wf:
        print("--- Original File Parameters ---")
        print(f"Channels: {wf.getnchannels()}")
        print(f"Sample Rate: {wf.getframerate()} Hz")
        print(f"Sample Width: {wf.getsampwidth()} bytes")
        print(f"Duration: {wf.getnframes() / wf.getframerate():.2f}s")

display_params(AUDIO_FILE_PATH)

--- Original File Parameters ---
Channels: 1
Sample Rate: 48000 Hz
Sample Width: 2 bytes
Duration: 2.43s
'display_params' ran in 0.0008 seconds


### Предобработка данных

In [20]:
# --- 2. Format Conversion with Tempfile ---
# Using tempfile.NamedTemporaryFile ensures the file is handled by the OS temp dir
temp_wav = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
PROCESSED_FILE_PATH: str = temp_wav.name
temp_wav.close() # Close handle so pydub/vosk can open it

def prepare_audio(input_path: str, output_path: str) -> None:
    audio = AudioSegment.from_file(input_path)
    # Vosk-friendly: Mono, 16000Hz, 16-bit
    audio = audio.set_frame_rate(16000).set_channels(1).set_sample_width(2)
    audio.export(output_path, format="wav")
    print(f"\n[INFO] Audio converted to temporary file: {output_path}")

prepare_audio(AUDIO_FILE_PATH, PROCESSED_FILE_PATH)


[INFO] Audio converted to temporary file: C:\Users\vadim\AppData\Local\Temp\tmplklx0i9d.wav


### Выполнение STT

In [21]:
# --- 3. Perform STT ---
@timer
def run_stt(audio_path: str) -> str:
    wf = wave.open(audio_path, "rb")
    
    if USE_VOCABULARY:
        rec = KaldiRecognizer(model, wf.getframerate(), json.dumps(CUSTOM_VOCABULARY, ensure_ascii=False))
    else:
        rec = KaldiRecognizer(model, wf.getframerate())

    rec.SetWords(True)

    results = []
    while True:
        data = wf.readframes(4000)
        if len(data) == 0:
            break
        if rec.AcceptWaveform(data):
            part = json.loads(rec.Result())
            results.append(part.get("text", ""))

    final_part = json.loads(rec.FinalResult())
    results.append(final_part.get("text", ""))

    return " ".join(results).strip()

print("\nStarting STT process...")
stt_result = run_stt(PROCESSED_FILE_PATH)
stt_result


Starting STT process...
'run_stt' ran in 1.0407 seconds


'табличку для инверсии пожалуйста'

### Очистка

In [22]:
# --- 5. Cleanup ---
try:
    os.remove(PROCESSED_FILE_PATH)
    print(f"\n[CLEANUP] Deleted temporary file: {PROCESSED_FILE_PATH}")
except OSError as e:
    print(f"Error deleting temp file: {e}")


[CLEANUP] Deleted temporary file: C:\Users\vadim\AppData\Local\Temp\tmplklx0i9d.wav
